Simulation of isotopic composition using Bolin number


In [1]:
import numpy as np

from PySDM import Formulae
from PySDM.physics import si
from PySDM.dynamics.isotopic_fractionation import HEAVY_ISOTOPES
from examples.PySDM_examples.Zaba_et_al.simulation import Simulation

In [2]:
isotope = '2H'
light_isotope = '1H'
atoms_per_light_molecule = 2
R_vap = 0
R_liq = 0

attributes = {
    "dry volume": np.nan,
    "kappa times dry volume": np.nan,
    **{f"moles_{isotope}": 0.0 * si.mole for isotope in HEAVY_ISOTOPES},
    "multiplicity": 1 * np.ones(1),
    "signed water mass": np.nan
}

formulae = Formulae(
        isotope_equilibrium_fractionation_factors="VanHook1968",
        # isotope_kinetic_fractionation_factors="JouzelAndMerlivat1984",
        isotope_relaxation_timescale="ZabaEtAl",
        isotope_diffusivity_ratios="HellmannAndHarvey2020",
        # isotope_ratio_evolution="ZabaEtAl", 
        # isotope_ventilation_ratio="Neglect",
        # isotope_meteoric_water_line="BarkanAndLuz2007+Dansgaard1964",
        # drop_growth="Mason1971",
    )

In [3]:
particulator, initial_conc_vap = Simulation.make_particulator(
    ff=formulae,
    molecular_isotopic_ratio=R_liq,
    initial_R_vap=R_vap,
    attributes=attributes,
    n_sd=1,
    dv=np.nan,
    dt=1*si.s,
    relative_humidity=np.nan,
    T=np.nan,
    isotope=isotope,  
)


/Users/agnieszkazaba/PycharmProjects/PySDM/PySDM/backends/numba.py:57: UserWarning: Disabling Numba threading due to ARM64 CPU (atomics do not work yet)
  warnings.warn(


In [5]:

isotopic_fraction = formulae.trivia.isotopic_fraction(
    particulator.environment[f"molality {isotope} in dry air"][0],
    particulator.environment["rhod"][0],
    initial_conc_vap,
)
R_vap = formulae.trivia.isotopic_ratio_assuming_single_heavy_isotope(
    isotopic_fraction=isotopic_fraction
)

initial_R_liq = (
    particulator.attributes[f"moles_{isotope}"][0]
    / particulator.attributes[f"moles_{light_isotope}"][0]
)

particulator.dynamics["Condensation"]()
particulator.dynamics["IsotopicFractionation"]()

KeyError: 'thd'